# Kia Soul: Decision Tree Baseline

This notebook trains a first model to classify CAN messages as normal (0) or attack (1). It uses the Kia Soul dataset and evaluates on held-out recordings.

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

## Load the data

The Parquet file is in the project root. Each row represents one CAN message.

In [ ]:
df = pd.read_parquet("../../kia_soul_features_all.parquet")

print(f"Dataset shape: {df.shape}")
df.head()

## Check the labels

`label_bin` is the correct answer: 0 means normal and 1 means attack. These counts include the complete dataset.

In [ ]:
class_counts = df["label_bin"].value_counts().sort_index()

print("Message counts:")
print(class_counts)

## Select the training partition

Keep the official test partition separate. Model development uses only rows marked `train`.

In [ ]:
train_df = df[df["split"] == "train"].copy()

print(f"Training messages: {len(train_df):,}")
print(train_df["label_bin"].value_counts().sort_index())

## Split training and validation recordings

Group by recording so the same file cannot be used for both training and validation. This split uses two validation recordings. They contain Injection and Spoofing contexts, so validation does not cover every attack family.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
train_idx, val_idx = next(splitter.split(train_df, groups=train_df["recording_id"]))

model_train = train_df.iloc[train_idx]
validation = train_df.iloc[val_idx]

print("Training:", len(model_train), "messages,", model_train["recording_id"].nunique(), "recordings")
print("Validation:", len(validation), "messages,", validation["recording_id"].nunique(), "recordings")
print("Validation attack contexts:", validation["attack_type"].unique())

## Select inputs and target

Use the message identifier, its length, and the eight byte columns as inputs. The target is `label_bin`. Source metadata and labels are excluded from the inputs.

In [ ]:
features = ["can_id_dec", "dlc", "byte_0", "byte_1", "byte_2", "byte_3",
            "byte_4", "byte_5", "byte_6", "byte_7"]

X_train = model_train[features]
y_train = model_train["label_bin"]

X_val = validation[features]
y_val = validation["label_bin"]

print("Training inputs:", X_train.shape)
print("Validation inputs:", X_val.shape)
X_train.head()

## Fill empty byte positions

Use zero padding for the selected inputs. The message length is retained so the model can distinguish shorter messages.

In [ ]:
X_train = X_train.fillna(0)
X_val = X_val.fillna(0)

## Train the decision tree

Limit tree depth to six as a simple starting point. The model learns from the training messages and labels.

In [ ]:
model = DecisionTreeClassifier(max_depth=6, random_state=42)
model.fit(X_train, y_train)

## Predict on validation data

Compare a few predictions with their true labels. Validation recordings were not used to fit the tree.

In [ ]:
predictions = model.predict(X_val)

print("Predictions:", predictions[:10])
print("Actual:     ", y_val.iloc[:10].to_numpy())

## Evaluate predictions

The confusion matrix shows correct predictions and errors. The report shows precision, recall, F1, and support. Undefined metrics are displayed as zero. These results describe this validation split only; the official test data remains unused.

In [ ]:
print("Confusion matrix:")
print(confusion_matrix(y_val, predictions))

print("\nClassification report:")
print(classification_report(y_val, predictions, target_names=["Normal", "Attack"], zero_division=0))